In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense
from tensorflow.keras.utils import to_categorical
import random

In [ ]:

with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read().lower()

print("Text Loaded Successfully!\n")

Text Loaded Successfully!



In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])

total_words = len(tokenizer.word_index) + 1
print("Total Words in Vocabulary:", total_words)


Total Words in Vocabulary: 518


In [ ]:
input_sequences = []

for line in text.split("\n"):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

max_seq_len = max(len(seq) for seq in input_sequences)

input_sequences = pad_sequences(input_sequences,
                                maxlen=max_seq_len,
                                padding='pre')

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

y = to_categorical(y, num_classes=total_words)

print("Training sequences created!\n")


Training sequences created!



In [ ]:
rnn_model = Sequential()
rnn_model.add(Embedding(total_words, 50, input_length=max_seq_len-1))
rnn_model.add(SimpleRNN(100))
rnn_model.add(Dense(total_words, activation='softmax'))

rnn_model.compile(loss='categorical_crossentropy',
                  optimizer='adam',
                  metrics=['accuracy'])

print("RNN Model Built!\n")

RNN Model Built!



/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
lstm_model = Sequential()
lstm_model.add(Embedding(total_words, 50, input_length=max_seq_len-1))
lstm_model.add(LSTM(100))
lstm_model.add(Dense(total_words, activation='softmax'))

lstm_model.compile(loss='categorical_crossentropy',
                   optimizer='adam',
                   metrics=['accuracy'])

print("LSTM Model Built!\n")

LSTM Model Built!



In [ ]:
print("Training RNN Model...")
rnn_model.fit(X, y, epochs=20, verbose=1)

print("\nTraining LSTM Model...")
lstm_model.fit(X, y, epochs=20, verbose=1)


Training RNN Model...
Epoch 1/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.0156 - loss: 6.2482
Epoch 2/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.0346 - loss: 6.0874
Epoch 3/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.0389 - loss: 5.8909
Epoch 4/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.0455 - loss: 5.8570
Epoch 5/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.0435 - loss: 5.7821
Epoch 6/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.0367 - loss: 5.7735
Epoch 7/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.0429 - loss: 5.7622
Epoch 8/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.0345 - loss: 5.6664
Epoch 9/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.0402 - loss: 5.6003
Epoch 10/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.0584 - loss: 5.4276
Epoch 11/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.0707 - loss: 5.2651
Epoch 12/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/ste

In [ ]:
def generate_text(model, seed_text, next_words=20):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list],
                                   maxlen=max_seq_len-1,
                                   padding='pre')

        predicted = model.predict(token_list, verbose=0)
        predicted_word_index = np.argmax(predicted)

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_word_index:
                output_word = word
                break

        seed_text += " " + output_word

    return seed_text


In [ ]:
seed = random.choice(text.split("\n"))
print("\nSeed Text:", seed)

print("\n--- RNN Generated Text ---")
print(generate_text(rnn_model, seed, 20))

print("\n--- LSTM Generated Text ---")
print(generate_text(lstm_model, seed, 20))


Seed Text: the celtics wear green and white uniforms.

--- RNN Generated Text ---
the celtics wear green and white uniforms. most april eastern and asian athletes contribute significantly anticipation defined their talent offensive brilliance viewing matchup style leagues the the

--- LSTM Generated Text ---
the celtics wear green and white uniforms. the the the the the the the the the the nba league the the the the the the the the
